In [5]:
import random
import numpy as np

from torchvision import datasets
from torchvision import transforms

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [6]:
# Sanity check: verify class labels and number of images per split
train_ds = datasets.ImageFolder("hair-dataset/data/train")
eval_ds = datasets.ImageFolder("hair-dataset/data/test")

print(train_ds.class_to_idx)          # {'curly': 0, 'straight': 1}
print(len(train_ds), len(eval_ds))    # 800 201

{'curly': 0, 'straight': 1}
800 201


##### Reproducibility rules

In [7]:
# Reproducibility: fix all random seeds so every run gives the same results
SEED = 42
random.seed(SEED)                         
np.random.seed(SEED)                        
torch.manual_seed(SEED)                     

# Force deterministic algorithms (no run-to-run variation)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True   # only relevant on GPU
torch.backends.cudnn.benchmark = False      # don't auto-pick fastest (non-deterministic) algorithms

# Train on CPU, as required for the reference results
device = torch.device("cpu")

##### Model and preprocessing

In [8]:
# Preprocessing: resize to 200x200, convert to tensor, normalize with ImageNet stats
# Used for both training (baseline, no augmentation) and evaluation
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),                      # PIL image -> tensor with values in [0, 1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],             # per-channel mean (R, G, B)
        std=[0.229, 0.224, 0.225],              # per-channel std (R, G, B)
    ),
])

# Load the fixed train and held-out evaluation splits (labels: curly=0, straight=1)
train_dataset = datasets.ImageFolder("hair-dataset/data/train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder("hair-dataset/data/test", transform=evaluation_transform)

In [9]:
# Fixed generator so the training data is shuffled the same way every run
loader_generator = torch.Generator().manual_seed(SEED)

# Training loader: shuffled in batches of 20, single process for reproducibility
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)

# Evaluation loader: no shuffling needed
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [10]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolution: 3 input channels (RGB) -> 32 feature maps, 3x3 filter
        # Output size: 200 - 3 + 1 = 198 -> 32 x 198 x 198
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        # Max pooling halves the size: 198 -> 99 -> 32 x 99 x 99
        self.pool = nn.MaxPool2d(kernel_size=2)
        # Fully connected layer: flattened features -> 64 neurons
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        # Output layer: 1 value (logit) for binary classification
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))  # conv -> ReLU -> pool
        x = x.flatten(start_dim=1)                        # flatten to (batch, features)
        x = nn.functional.relu(self.hidden(x))            # hidden layer + ReLU
        return self.output(x)                             # raw logits, no sigmoid here

# Create the model after the seeds so the initial weights are reproducible
model = HairModel().to(device)

In [11]:
# Binary cross-entropy loss that applies sigmoid internally (expects logits)
criterion = nn.BCEWithLogitsLoss()

# Stochastic gradient descent with momentum
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,        # learning rate
    momentum=0.8,    # smooths updates using previous gradients
)

##### Question 1 — loss and output contract

In [12]:
# Check: the model outputs one logit per image
images, labels = next(iter(train_loader))
outputs = model(images.to(device))
print(outputs.shape)    # expected: torch.Size([20, 1]) -> 1 logit per image

# The loss works directly on logits; labels must be float with the same shape
loss = criterion(outputs, labels.float().unsqueeze(1).to(device))
print(loss.item())

torch.Size([20, 1])
0.7192039489746094


In [13]:
# Question 1: test all four loss options on the actual model output
images, labels = next(iter(evaluation_loader))
with torch.no_grad():
    logits = model(images.to(device))       # raw model output
y = labels.float().unsqueeze(1).to(device)  # labels as float, shape [20, 1]

print("Model output shape:", logits.shape)  # [20, 1] -> one logit per image

# 1) BCEWithLogitsLoss: should equal sigmoid + binary cross-entropy computed by hand
bce = nn.BCEWithLogitsLoss()(logits, y)
p = torch.sigmoid(logits)
manual_bce = -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()
print("\n[BCEWithLogitsLoss]")
print("  PyTorch:", bce.item())
print("  By hand:", manual_bce.item())     # same value -> loss does sigmoid + BCE

# 2) CrossEntropyLoss: expects one logit PER CLASS (2 outputs), we only have 1
ce = nn.CrossEntropyLoss()
print("\n[CrossEntropyLoss]")
print("  Labels all 0:", ce(logits, torch.zeros(20, dtype=torch.long)).item())  # always 0.0 -> nothing to learn
try:
    ce(logits, torch.ones(20, dtype=torch.long))                               # class 1 doesn't exist
except Exception as e:
    print("  Labels all 1: ERROR ->", e)

# 3) MSELoss: runs, but treats the logit as a regression value
#    A very confident, CORRECT prediction (logit 10 for 'straight') gets punished hard
logit, target = torch.tensor([[10.0]]), torch.tensor([[1.0]])
print("\n[MSELoss vs BCEWithLogitsLoss for a confident correct prediction]")
print("  MSE:", nn.MSELoss()(logit, target).item())             # 81.0 -> wrongly punished
print("  BCE:", nn.BCEWithLogitsLoss()(logit, target).item())   # ~0.00005 -> correctly rewarded

# 4) CosineEmbeddingLoss: needs two embeddings + a similarity label, not a classifier output
print("\n[CosineEmbeddingLoss]")
try:
    nn.CosineEmbeddingLoss()(logits, y)
except Exception as e:
    print("  ERROR ->", e)

Model output shape: torch.Size([20, 1])

[BCEWithLogitsLoss]
  PyTorch: 0.6279961466789246
  By hand: 0.6279962658882141

[CrossEntropyLoss]
  Labels all 0: 0.0
  Labels all 1: ERROR -> Target 1 is out of bounds.

[MSELoss vs BCEWithLogitsLoss for a confident correct prediction]
  MSE: 81.0
  BCE: 4.5398901420412585e-05

[CosineEmbeddingLoss]
  ERROR -> CosineEmbeddingLoss.forward() missing 1 required positional argument: 'target'
